<a href="https://colab.research.google.com/github/adriandarmali/pacezero-lp-engine/blob/main/PaceZero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set Up

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import re
import json
import time
import sqlite3
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from google.colab import auth, userdata, drive
from google import genai
from google.genai import types
import gspread
from google.auth import default

In [5]:
# --- Paths ---
DRIVE_BASE   = '/content/drive/MyDrive/PaceZero'
DB_PATH      = f'{DRIVE_BASE}/lp_pipeline.db'
CSV_PATH     = f'{DRIVE_BASE}/challenge_contacts.csv'   # <-- update if needed
EXPORT_PATH  = f'{DRIVE_BASE}/scored_results.csv'

# --- Scoring weights (from spec) ---
WEIGHTS = {
    'sector_fit':         0.35,
    'relationship_depth': 0.30,
    'halo_value':         0.20,
    'emerging_fit':       0.15,
}

# --- Tier thresholds ---
TIERS = [
    (8.0, 'PRIORITY CLOSE'),
    (6.5, 'STRONG FIT'),
    (5.0, 'MODERATE FIT'),
    (0.0, 'WEAK FIT'),
]

In [6]:
import pandas as pd

# Update this path to match where your file is in Drive
df = pd.read_csv('/content/drive/MyDrive/challenge_contacts.csv')
df.head()

,Contact Name,Organization,Org Type,Role,Email,Region,Contact Status,Relationship Depth
0,Kim Lew,Columbia Investment Management Company,Endowment,CEO,NaN,NYC,New Contact,7
1,Lan Cai,PBUCC,Endowment,CIO,NaN,NYC,New Contact,7
2,Saeed Mouzaffar,Willett Advisors,Single Family Office,Director,NaN,NYC,New Contact,4
3,Michael FitzSimons,Bessemer Trust,Multi-Family Office,"Managing Director, Head of Alternatives Advisory",NaN,NYC,New Contact,4
4,David Sheng,Aksia,RIA/FIA,Investment Consultant,david.sheng@aksia.com,NYC,New Contact,4


# CELL 2: IMPORTS & CONFIGURATION


In [7]:
# --- Ground-truth calibration anchors (from PDF) ---
CALIBRATION_ANCHORS = {
    "The Rockefeller Foundation":           {"sector_fit": 9, "halo_score": 9, "emerging_fit": 8},
    "Pension Boards United Church of Christ":{"sector_fit": 8, "halo_score": 6, "emerging_fit": 8},
    "PBUCC":                                {"sector_fit": 8, "halo_score": 6, "emerging_fit": 8},
    "Inherent Group":                       {"sector_fit": 8, "halo_score": 3, "emerging_fit": 5},
    "Meridian Capital Group LLC":           {"sector_fit": 1, "halo_score": 3, "emerging_fit": 1},
}

# --- Check size allocation ranges by org type ---
ALLOCATION_PCT = {
    'Pension':              (0.005, 0.02),
    'Insurance':            (0.005, 0.02),
    'Endowment':            (0.01,  0.03),
    'Foundation':           (0.01,  0.03),
    'Fund of Funds':        (0.02,  0.05),
    'Multi-Family Office':  (0.02,  0.05),
    'Single Family Office': (0.03,  0.10),
    'HNWI':                 (0.03,  0.10),
    'Asset Manager':        (0.005, 0.03),
    'RIA/FIA':              (0.005, 0.03),
    'Private Capital Firm': (0.005, 0.03),
}

print("✅ Configuration loaded.")

✅ Configuration loaded.


# Mount Google Drive and Create Database

In [8]:
drive.mount('/content/drive')
os.makedirs(DRIVE_BASE, exist_ok=True)

def get_db():
    """Return a SQLite connection to the persistent database on Drive."""
    return sqlite3.connect(DB_PATH)

def init_db():
    """Create all tables if they don't exist."""
    con = get_db()
    cur = con.cursor()

    # One row per unique organization — enrichment + scores
    cur.execute("""
        CREATE TABLE IF NOT EXISTS organizations (
            org_name                TEXT PRIMARY KEY,
            org_type_csv            TEXT,
            org_type_confirmed      TEXT,
            is_lp                   INTEGER,
            is_gp_or_service_provider INTEGER,
            gp_evidence             TEXT,
            org_description         TEXT,
            aum_estimate            TEXT,
            aum_figure_millions     REAL,
            external_fund_allocations TEXT,
            sustainability_mandate  TEXT,
            brand_recognition       TEXT,
            emerging_manager_appetite TEXT,
            sector_fit_score        REAL,
            sector_fit_reasoning    TEXT,
            halo_score              REAL,
            halo_reasoning          TEXT,
            emerging_fit_score      REAL,
            emerging_fit_reasoning  TEXT,
            confidence              TEXT,
            data_quality_notes      TEXT,
            check_size_low_k        REAL,
            check_size_high_k       REAL,
            enrichment_raw          TEXT,
            enriched_at             TEXT,
            tokens_used             INTEGER,
            anomaly_flags           TEXT
        )
    """)

    # One row per contact — links to organizations
    cur.execute("""
        CREATE TABLE IF NOT EXISTS contacts (
            id                  INTEGER PRIMARY KEY AUTOINCREMENT,
            contact_name        TEXT,
            org_name            TEXT,
            org_type            TEXT,
            role                TEXT,
            email               TEXT,
            region              TEXT,
            contact_status      TEXT,
            relationship_depth  REAL,
            composite_score     REAL,
            tier                TEXT,
            is_test_record      INTEGER DEFAULT 0,
            scored_at           TEXT,
            FOREIGN KEY (org_name) REFERENCES organizations(org_name)
        )
    """)

    # One row per pipeline run — cost & metadata tracking
    cur.execute("""
        CREATE TABLE IF NOT EXISTS scoring_runs (
            run_id          INTEGER PRIMARY KEY AUTOINCREMENT,
            started_at      TEXT,
            completed_at    TEXT,
            orgs_processed  INTEGER,
            orgs_skipped    INTEGER,
            total_tokens    INTEGER,
            estimated_cost_usd REAL,
            notes           TEXT
        )
    """)

    # Anomaly log
    cur.execute("""
        CREATE TABLE IF NOT EXISTS anomaly_flags (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            org_name    TEXT,
            flag_type   TEXT,
            description TEXT,
            flagged_at  TEXT
        )
    """)

    con.commit()
    con.close()
    print("✅ Database initialised at:", DB_PATH)

init_db()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Database initialised at: /content/drive/MyDrive/PaceZero/lp_pipeline.db


# Test Train Split

In [9]:
# Test orgs — held out for calibration validation
TEST_ORGS = [
    "PBUCC",
    "Pension Boards United Church of Christ",
    "Meridian Capital Group LLC",
    "Inherent Group",
    "The Rockefeller Foundation",
]

# Split
test_df  = df[df['Organization'].isin(TEST_ORGS)].copy()
train_df = df[~df['Organization'].isin(TEST_ORGS)].copy()

# Sanity check
assert not train_df['Organization'].isin(TEST_ORGS).any(), "⚠️ Test data leaked into training!"

print(f"✅ Training set : {len(train_df)} contacts, {train_df['Organization'].nunique()} unique orgs")
print(f"✅ Test set     : {len(test_df)} contacts, {test_df['Organization'].nunique()} unique orgs")

✅ Training set : 95 contacts, 89 unique orgs
✅ Test set     : 5 contacts, 5 unique orgs


## Scoring Prompt

In [10]:
SCORING_PROMPT = """
You are an LP analyst for PaceZero Capital Partners — a Toronto-based
sustainability-focused private credit firm (Fund II, emerging manager).

Research the organization below using web search, then score it across
3 dimensions using the rubrics provided.

ORGANIZATION:
  Name   : {org_name}
  Type   : {org_type}
  Contact: {role}

CRITICAL RULE — LP vs GP DISTINCTION:
  LP = allocates capital INTO externally managed funds → score normally
  GP = PRIMARILY manages funds for others / originates loans / brokers deals → apply caps

  ⚠️  IMPORTANT: Many large institutions do BOTH.
  If an org allocates to external managers even partially, treat as LP.
  Only mark is_gp_or_service_provider = true if the org is PURELY a:
    - Real estate broker or advisory firm
    - Loan originator with no external fund allocations
    - Pure investment bank or placement agent
    - Pure service provider with no investing activity

  Examples of LPs (do NOT flag as GP):
    - Neuberger Berman              → asset manager that allocates to external funds
    - Lincoln Financial             → insurance company with large LP allocation programme
    - Bessemer Trust                → MFO that allocates to external managers
    - Brown Brothers Harriman       → private bank with LP allocation activity
    - Ludwig Institute for Cancer Research → medical foundation with investment office
    - Safra Group                   → private banking group with external allocations

  Examples of true GPs/service providers (flag as GP):
    - Meridian Capital Group        → pure CRE brokerage, no external fund allocations
    - A placement agent or fundraising consultant
    - A company that only originates and holds loans

  GP hard caps (only apply if PURELY a GP/service provider):
    sector_fit ≤ 2, emerging_fit ≤ 2, halo ≤ 3

RUBRIC 1 — SECTOR & MANDATE FIT (1–10):
  9–10 : Private credit allocation + sustainability/ESG mandate (BOTH confirmed)
  7–8  : One confirmed strongly, other strongly implied
  5–6  : LP with alternatives exposure but weak ESG, OR strong ESG but no credit signal
  3–4  : LP confirmed but no alignment signals found
  1–2  : Purely a GP / broker / lender / service provider ← hard cap

RUBRIC 2 — HALO & STRATEGIC VALUE (1–10):
  9–10 : Globally recognized — Rockefeller, major university endowment tier
  7–8  : Well-known specifically in impact/sustainability LP circles
  5–6  : Respected regionally or within a niche; some public presence
  3–4  : Limited public presence; small or private institution
  1–2  : Unknown, purely private, no meaningful public footprint

  SFO GUIDANCE:
    Most SFOs score 3–4 (private, limited public presence)
    Exception: score 6–7 if the SFO is explicitly tied to a famous family
               or billionaire AND manages $1B+ (e.g. Bloomberg, Rockefeller)
    Never score an SFO above 7 unless globally famous at institutional level

  Pure GP/service provider hard cap → halo ≤ 3

RUBRIC 3 — EMERGING MANAGER FIT (1–10):
  9–10 : Documented emerging manager programme; has backed Fund I/II on record
  7–8  : Flexible smaller institution; SFO or small MFO with open mandate
         and no evidence of manager size restrictions
  5–6  : Mid-size MFO or FoF with no explicit programme but typically flexible
  3–4  : Large institutional allocator ($10B+ AUM); likely prefers established managers
  1–2  : Known policy against emerging managers OR purely a GP/service provider

  ⚠️  Do NOT default everything to 4–5. Actively look for evidence
      in both directions. SFOs and small MFOs should generally score
      6–7 unless there is evidence of restrictions.
  ⚠️  Pure GP/service provider hard cap → emerging_fit ≤ 2

Respond ONLY with valid JSON. No preamble, no markdown, no extra text.

{{
  "org_name": "{org_name}",
  "org_description": "<2 sentence factual summary>",
  "is_lp": <true|false>,
  "is_gp_or_service_provider": <true|false>,
  "gp_evidence": "<why flagged as GP or null if LP>",
  "aum_estimate": "<e.g. '$2.5B' or 'Unknown'>",
  "aum_figure_millions": <number or null>,
  "sector_fit_score": <1-10>,
  "sector_fit_reasoning": "<2 sentences citing evidence>",
  "halo_score": <1-10>,
  "halo_reasoning": "<2 sentences citing evidence>",
  "emerging_fit_score": <1-10>,
  "emerging_fit_reasoning": "<2 sentences citing evidence>",
  "confidence": "<HIGH|MEDIUM|LOW>",
  "data_quality_notes": "<what was or wasn't available>"
}}
"""

print("✅ Prompt updated")

✅ Prompt updated


In [22]:
!pip install openai --quiet

from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('openai_key'))
MODEL = "gpt-4o"  # cheapest capable model

def score_org(org_name, org_type, role):
    prompt = SCORING_PROMPT.format(
        org_name=org_name,
        org_type=org_type,
        role=role,
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        max_tokens=1000,
    )

    raw_text = response.choices[0].message.content.strip()

    # Track input and output separately
    input_tokens  = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    data = json.loads(raw_text)
    data['_input_tokens']  = input_tokens
    data['_output_tokens'] = output_tokens
    data['_tokens']        = input_tokens + output_tokens

    return data

print("✅ score_org updated with token tracking")


# ── Quick test ──
test_result = score_org(
    org_name = "Carnegie Corporation of New York",
    org_type = "Foundation",
    role     = "Senior Director"
)

print(f"Org         : {test_result.get('org_name')}")
print(f"Description : {test_result.get('org_description')}")
print(f"AUM         : {test_result.get('aum_estimate')}")
print(f"Is LP       : {test_result.get('is_lp')}")
print(f"Is GP       : {test_result.get('is_gp_or_service_provider')}")
print(f"Sector Fit  : {test_result.get('sector_fit_score')} — {test_result.get('sector_fit_reasoning')}")
print(f"Halo        : {test_result.get('halo_score')} — {test_result.get('halo_reasoning')}")
print(f"Emerging    : {test_result.get('emerging_fit_score')} — {test_result.get('emerging_fit_reasoning')}")
print(f"Confidence  : {test_result.get('confidence')}")
print(f"Tokens used : {test_result.get('_tokens'):,}")

✅ score_org updated with token tracking
Org         : Carnegie Corporation of New York
Description : Carnegie Corporation of New York is a philanthropic foundation established by Andrew Carnegie to advance the diffusion of knowledge and understanding. It focuses on the advancement of education and knowledge, particularly in the areas of teacher preparation, civic education, and immigrant integration.
AUM         : $3.5B
Is LP       : True
Is GP       : False
Sector Fit  : 5 — Carnegie Corporation is primarily focused on education and knowledge dissemination, with limited indication of private credit or specific ESG mandates.
Halo        : 9 — The foundation is globally recognized, founded by Andrew Carnegie, and holds significant influence in philanthropic and educational circles.
Emerging    : 3 — As a large institutional allocator, it is likely to prefer established managers with no documented focus on emerging managers.
Confidence  : MEDIUM
Tokens used : 1,401


## Training

In [23]:
import time

results = []
failed  = []

unique_orgs = train_df.groupby('Organization').first().reset_index()
print(f"🚀 Scoring {len(unique_orgs)} unique orgs...\n")

for _, row in unique_orgs.iterrows():
    org_name = row['Organization']
    org_type = row['Org Type']
    role     = row['Role']

    try:
        print(f"🔍 {org_name}")
        data = score_org(org_name, org_type, role)
        data['org_name'] = org_name
        data['org_type'] = org_type
        results.append(data)

    except Exception as e:
        print(f"  ❌ Failed: {e}")
        failed.append(org_name)

    time.sleep(1)

print(f"\n✅ Scored {len(results)} orgs successfully")
print(f"❌ Failed: {len(failed)}")

🚀 Scoring 89 unique orgs...

🔍 Aeterna Capital
🔍 Aksia
🔍 AlTi Global
🔍 Alpha Square Group
🔍 Amadeus Asset
🔍 Anholt Services
🔍 Aysen Corporation
🔍 BBR Partners
🔍 Bessemer Trust
🔍 Brown Brothers Harriman & Co.
🔍 Camino Partners
🔍 Carilion Clinic
🔍 Carnegie Corporation of New York
🔍 Catal Group
🔍 CoVision Advisors
🔍 Collaborative Capital Advisors
🔍 Columbia Investment Management Company
🔍 DV Trading
🔍 Diag Partners
🔍 Dominion Capital
🔍 Ellis Lake Investment Management
🔍 FBE Limited (Fruchthandler Brothers)
🔍 First New York
🔍 Flat World Partners
🔍 Gratitude Railroad
🔍 HSBC
🔍 Helmsley Charitable Trust
🔍 Inlihtan Capital
🔍 Invus Financial Advisors, LLC
🔍 Itau USA Asset Management
🔍 JMC Family Office
🔍 JOIA Investments LLC
🔍 Jindalee LLC
🔍 Johnson Family Office
🔍 Kazazian Asset Management
🔍 Kemnay
🔍 Laviv Family Office
🔍 Lepercq de Neuflize Asset Management
🔍 Lincoln Financial Group
🔍 Logiq
🔍 Ludwig Institute for Cancer Research
🔍 Lufin Americas LLC
🔍 Luminos Capital
🔍 MBX Group
🔍 Magnitude C

In [70]:
WEIGHTS = {
    'sector_fit':         0.35,
    'relationship_depth': 0.30,
    'halo_value':         0.20,
    'emerging_fit':       0.15,
}

TIERS = [
    (8.0, 'PRIORITY CLOSE'),
    (6.5, 'STRONG FIT'),
    (5.0, 'MODERATE FIT'),
    (0.0, 'WEAK FIT'),
]

def compute_composite(sector_fit, relationship_depth, halo_value, emerging_fit):
    return round(
        sector_fit         * WEIGHTS['sector_fit']         +
        relationship_depth * WEIGHTS['relationship_depth'] +
        halo_value         * WEIGHTS['halo_value']         +
        emerging_fit       * WEIGHTS['emerging_fit'],
        2
    )

def classify_tier(composite):
    for threshold, label in TIERS:
        if composite >= threshold:
            return label
    return 'WEAK FIT'


# ── Build scored dataframe ──
scored_rows = []

for _, contact in train_df.iterrows():
    org_name = contact['Organization']

    # Find enrichment result for this org
    org_data = next((r for r in results if r['org_name'] == org_name), None)
    if not org_data:
        continue

    sf  = org_data.get('sector_fit_score', 5)
    ha  = org_data.get('halo_score', 5)
    em  = org_data.get('emerging_fit_score', 5)
    rel = float(contact['Relationship Depth'])

    composite = compute_composite(sf, rel, ha, em)
    tier      = classify_tier(composite)

    scored_rows.append({
        'Contact Name':         contact['Contact Name'],
        'Organization':         org_name,
        'Org Type':             contact['Org Type'],
        'Role':                 contact['Role'],
        'Region':               contact['Region'],
        'Contact Status':       contact['Contact Status'],
        'Sector Fit (D1)':      sf,
        'Rel Depth (D2)':       rel,
        'Halo Value (D3)':      ha,
        'Emerging Fit (D4)':    em,
        'Composite Score':      composite,
        'Tier':                 tier,
        'AUM Estimate':         org_data.get('aum_estimate', 'Unknown'),
        'Is LP':                org_data.get('is_lp'),
        'Is GP':                org_data.get('is_gp_or_service_provider'),
        'Confidence':           org_data.get('confidence'),
        'Org Description':      org_data.get('org_description'),
        'Sector Fit Reasoning': org_data.get('sector_fit_reasoning'),
        'Halo Reasoning':       org_data.get('halo_reasoning'),
        'Emerging Reasoning':   org_data.get('emerging_fit_reasoning'),
    })

scored_df = pd.DataFrame(scored_rows).sort_values('Composite Score', ascending=False)

print(f"✅ Scored {len(scored_df)} contacts\n")
print(scored_df[['Contact Name', 'Organization', 'Composite Score', 'Tier']].to_string(index=False))

✅ Scored 94 contacts

       Contact Name                                   Organization  Composite Score           Tier
       Isaac Eskind                            Flat World Partners             8.45 PRIORITY CLOSE
Roman Torres Boscan                  The Schmidt Family Foundation             7.80     STRONG FIT
     Howard Fischer                             Gratitude Railroad             6.70     STRONG FIT
 Alexander Gottlieb                               Neuberger Berman             6.20   MODERATE FIT
         Zack Miles                                Anholt Services             6.15   MODERATE FIT
            Kim Lew         Columbia Investment Management Company             6.10   MODERATE FIT
      Antonio Casal                                    AlTi Global             5.75   MODERATE FIT
       Shane Wolter                                           HSBC             5.70   MODERATE FIT
             Al Kim                      Helmsley Charitable Trust             5.50   M

## Cross Validation

In [15]:
import numpy as np

WEIGHTS = {
    'sector_fit':         0.35,
    'relationship_depth': 0.30,
    'halo_value':         0.20,
    'emerging_fit':       0.15,
}

TIERS = [
    (8.0, 'PRIORITY CLOSE'),
    (6.5, 'STRONG FIT'),
    (5.0, 'MODERATE FIT'),
    (0.0, 'WEAK FIT'),
]

def compute_composite(sector_fit, relationship_depth, halo_value, emerging_fit):
    return round(
        sector_fit         * WEIGHTS['sector_fit']         +
        relationship_depth * WEIGHTS['relationship_depth'] +
        halo_value         * WEIGHTS['halo_value']         +
        emerging_fit       * WEIGHTS['emerging_fit'],
        2
    )

def classify_tier(composite):
    for threshold, label in TIERS:
        if composite >= threshold:
            return label
    return 'WEAK FIT'

def validate_training_results(results, train_df):
    print("=" * 60)
    print("TRAINING SET VALIDATION REPORT")
    print("=" * 60)

    issues = []

    # CHECK 1: Rule Consistency
    print("\nCHECK 1 -- Rule Consistency")
    print("-" * 40)

    for r in results:
        org      = r['org_name']
        sf       = r.get('sector_fit_score', 5)
        halo     = r.get('halo_score', 5)
        em       = r.get('emerging_fit_score', 5)
        is_gp    = r.get('is_gp_or_service_provider', False)
        org_type = r.get('org_type', '')
        conf     = r.get('confidence', 'MEDIUM')

        # GPs must score SF <= 2
        if is_gp and sf > 2:
            msg = f"WARNING: GP scored SF={sf} (must be <=2): {org}"
            print(msg); issues.append(msg)

        # GPs must score EM <= 2
        if is_gp and em > 2:
            msg = f"WARNING: GP scored EM={em} (must be <=2): {org}"
            print(msg); issues.append(msg)

        # GPs must score Halo <= 3
        if is_gp and halo > 3:
            msg = f"WARNING: GP scored Halo={halo} (must be <=3): {org}"
            print(msg); issues.append(msg)

        # SFOs should not score Halo > 7
        # allows famous SFOs like Bessemer, Willett to score 6-7
        if 'Family Office' in org_type and halo > 7:
            msg = f"WARNING: SFO scored Halo={halo} (expected <=7): {org}"
            print(msg); issues.append(msg)

        # Low confidence should not produce scores above 7
        if conf == 'LOW' and any(s > 7 for s in [sf, halo, em]):
            msg = f"WARNING: LOW confidence but high score SF={sf} Halo={halo} EM={em}: {org}"
            print(msg); issues.append(msg)

    if not issues:
        print("OK -- No rule violations found")


    # CHECK 2: Stability Test
    print("\nCHECK 2 -- Stability Test (3 orgs scored twice)")
    print("-" * 40)

    stability_orgs = [
        {"org_name": "Carnegie Corporation of New York", "org_type": "Foundation",         "role": "Senior Director"},
        {"org_name": "Bessemer Trust",                   "org_type": "Multi-Family Office", "role": "Managing Director"},
        {"org_name": "Neuberger Berman",                 "org_type": "Fund of Funds",       "role": "Senior Vice President"},
    ]

    for org in stability_orgs:
        print(f"\n  Stability: {org['org_name']}")
        try:
            run1 = score_org(org['org_name'], org['org_type'], org['role'])
            time.sleep(2)
            run2 = score_org(org['org_name'], org['org_type'], org['role'])

            for dim, key in [
                ('SF',   'sector_fit_score'),
                ('Halo', 'halo_score'),
                ('EM',   'emerging_fit_score')
            ]:
                s1    = run1.get(key, 5)
                s2    = run2.get(key, 5)
                delta = abs(s1 - s2)
                flag  = 'PASS' if delta <= 1.5 else 'FAIL'
                print(f"    {flag} {dim}: run1={s1} run2={s2} delta={delta}")

        except Exception as e:
            print(f"    ERROR: {e}")
        time.sleep(2)


    # CHECK 3: Score Distribution
    print("\nCHECK 3 -- Score Distribution")
    print("-" * 40)

    sf_scores   = [r.get('sector_fit_score', 5)  for r in results]
    halo_scores = [r.get('halo_score', 5)         for r in results]
    em_scores   = [r.get('emerging_fit_score', 5) for r in results]

    for label, scores in [
        ('Sector Fit',   sf_scores),
        ('Halo',         halo_scores),
        ('Emerging Fit', em_scores)
    ]:
        mean        = round(np.mean(scores), 2)
        std         = round(np.std(scores),  2)
        lo          = min(scores)
        hi          = max(scores)
        spread_flag = 'OK' if std >= 1.5 else 'WARNING -- low spread, prompt may not be discriminating enough'
        print(f"  {label:<15} mean={mean}  std={std}  range={lo}-{hi}  {spread_flag}")

    # Tier breakdown
    print("\n  Tier breakdown:")
    composite_scores = []
    for r in results:
        match = train_df[train_df['Organization'] == r['org_name']]
        if match.empty:
            continue
        rel       = float(match.iloc[0]['Relationship Depth'])
        sf        = r.get('sector_fit_score', 5)
        ha        = r.get('halo_score', 5)
        em        = r.get('emerging_fit_score', 5)
        composite = compute_composite(sf, rel, ha, em)
        composite_scores.append(composite)

    for threshold, label in TIERS:
        count = sum(1 for s in composite_scores if s >= threshold)
        bar   = '#' * count
        print(f"  {label:<20} {count:>3} {bar}")


    # Summary
    print("\n" + "=" * 60)
    print(f"Total issues found : {len(issues)}")
    if len(issues) == 0:
        print("OK -- Training set looks well calibrated, safe to run test set")
    elif len(issues) <= 3:
        print("MINOR -- Consider prompt tweaks before running test set")
    else:
        print("FAIL -- Significant issues, fix prompt before touching test set")

    return issues


issues = validate_training_results(results, train_df)

TRAINING SET VALIDATION REPORT

CHECK 1 -- Rule Consistency
----------------------------------------

CHECK 2 -- Stability Test (3 orgs scored twice)
----------------------------------------

  Stability: Carnegie Corporation of New York
    PASS SF: run1=5 run2=5 delta=0
    PASS Halo: run1=9 run2=9 delta=0
    PASS EM: run1=3 run2=3 delta=0

  Stability: Bessemer Trust
    PASS SF: run1=5 run2=5 delta=0
    FAIL Halo: run1=9 run2=7 delta=2
    PASS EM: run1=5 run2=5 delta=0

  Stability: Neuberger Berman
    PASS SF: run1=7 run2=7 delta=0
    PASS Halo: run1=9 run2=9 delta=0
    PASS EM: run1=5 run2=5 delta=0

CHECK 3 -- Score Distribution
----------------------------------------
  Sector Fit      mean=4.24  std=1.61  range=1-9  OK
  Halo            mean=4.21  std=1.81  range=3-9  OK
  Emerging Fit    mean=5.37  std=1.92  range=1-7  OK

  Tier breakdown:
  PRIORITY CLOSE         1 #
  STRONG FIT             2 ##
  MODERATE FIT          25 #########################
  WEAK FIT         

## Cross Validation Loss

In [16]:
def rule_violation_loss(results):
    total_checks     = 0
    total_violations = 0

    for r in results:
        is_gp    = r.get('is_gp_or_service_provider', False)
        sf       = r.get('sector_fit_score', 5)
        halo     = r.get('halo_score', 5)
        em       = r.get('emerging_fit_score', 5)
        org_type = r.get('org_type', '')

        total_checks += 4

        if is_gp and sf > 2:   total_violations += 1
        if is_gp and em > 2:   total_violations += 1
        if is_gp and halo > 3: total_violations += 1
        if 'Family Office' in org_type and halo > 7: total_violations += 1

    loss = round(total_violations / total_checks, 4)
    print(f"Rule violations  : {total_violations} / {total_checks}")
    print(f"Violation rate   : {loss:.2%}")
    print(f"Rule Loss        : {loss}  (0.0 = perfect, 1.0 = all rules broken)")
    return loss

l2 = rule_violation_loss(results)

Rule violations  : 1 / 356
Violation rate   : 0.28%
Rule Loss        : 0.0028  (0.0 = perfect, 1.0 = all rules broken)


In [17]:
# Input delta values from CHECK 2 stability test
stability_pairs_deltas = [
    # Carnegie
    {'sf': 0, 'halo': 0, 'em': 0},
    # Bessemer
    {'sf': 0, 'halo': 2, 'em': 0},
    # Neuberger
    {'sf': 0, 'halo': 0, 'em': 0},
]

all_deltas = []
for pair in stability_pairs_deltas:
    all_deltas.extend([pair['sf'], pair['halo'], pair['em']])

l3 = round(np.mean(all_deltas), 2)
print(f"Mean delta across all runs : {l3}")
print(f"Stability Loss             : {l3}  (0.0 = perfectly stable)")

Mean delta across all runs : 0.22
Stability Loss             : 0.22  (0.0 = perfectly stable)


In [18]:
# Anchor loss not included yet -- comes after test set
combined_pretrain_loss = round((l2 * 0.60) + (l3 * 0.40), 4)

print("=" * 50)
print("COMBINED PRE-TEST LOSS")
print("=" * 50)
print(f"  Rule Violation Loss (60%) : {l2}")
print(f"  Stability Loss      (40%) : {l3}")
print(f"  Combined Loss             : {combined_pretrain_loss}")
print()
if combined_pretrain_loss < 0.10:
    print("OK -- Loss acceptable, ready to run test set")
elif combined_pretrain_loss < 0.20:
    print("MINOR -- Acceptable but consider fixing remaining issues first")
else:
    print("FAIL -- Fix prompt before running test set")

COMBINED PRE-TEST LOSS
  Rule Violation Loss (60%) : 0.0028
  Stability Loss      (40%) : 0.22
  Combined Loss             : 0.0897

OK -- Loss acceptable, ready to run test set


## Test Set Calibration

In [19]:
CALIBRATION_ANCHORS = {
    "The Rockefeller Foundation":             {"sector_fit": 9, "halo": 9, "emerging_fit": 8},
    "Pension Boards United Church of Christ": {"sector_fit": 8, "halo": 6, "emerging_fit": 8},
    "PBUCC":                                  {"sector_fit": 8, "halo": 6, "emerging_fit": 8},
    "Inherent Group":                         {"sector_fit": 8, "halo": 3, "emerging_fit": 5},
    "Meridian Capital Group LLC":             {"sector_fit": 1, "halo": 3, "emerging_fit": 1},
}

test_results = []

unique_test_orgs = test_df.groupby('Organization').first().reset_index()
print(f"Running test set -- {len(unique_test_orgs)} orgs\n")

for _, row in unique_test_orgs.iterrows():
    org_name = row['Organization']
    try:
        print(f"Scoring: {org_name}")
        data = score_org(org_name, row['Org Type'], row['Role'])
        data['org_name'] = org_name
        test_results.append(data)
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(1)

print(f"\nDone -- {len(test_results)} orgs scored")

Running test set -- 5 orgs

Scoring: Inherent Group
Scoring: Meridian Capital Group LLC
Scoring: PBUCC
Scoring: Pension Boards United Church of Christ
Scoring: The Rockefeller Foundation

Done -- 5 orgs scored


In [20]:
def anchor_loss(test_results):
    errors = []

    print("=" * 65)
    print("TEST SET CALIBRATION REPORT")
    print("=" * 65)
    print(f"  {'Org':<40} {'SF':>3} {'exp':>4}  {'Halo':>4} {'exp':>4}  {'EM':>3} {'exp':>4}  MSE")
    print(f"  {'-'*63}")

    for r in test_results:
        anchor = CALIBRATION_ANCHORS.get(r['org_name'])
        if not anchor:
            continue

        sf_got = r.get('sector_fit_score', 5)
        ha_got = r.get('halo_score', 5)
        em_got = r.get('emerging_fit_score', 5)

        sf_exp = anchor['sector_fit']
        ha_exp = anchor['halo']
        em_exp = anchor['emerging_fit']

        sf_err  = (sf_got - sf_exp) ** 2
        ha_err  = (ha_got - ha_exp) ** 2
        em_err  = (em_got - em_exp) ** 2

        # Weighted by dimension importance
        mse = round((sf_err * 0.35) + (ha_err * 0.20) + (em_err * 0.15), 2)
        errors.append(mse)

        print(f"  {r['org_name'][:40]:<40} "
              f"{sf_got:>3}  {sf_exp:>3}  "
              f"{ha_got:>4}  {ha_exp:>3}  "
              f"{em_got:>3}  {em_exp:>3}  {mse}")

    l1 = round(np.mean(errors), 2)
    print(f"\n  Anchor MSE Loss : {l1}")
    print(f"  Anchor RMSE     : {round(np.sqrt(l1), 2)}")
    return l1

l1 = anchor_loss(test_results)

TEST SET CALIBRATION REPORT
  Org                                       SF  exp  Halo  exp   EM  exp  MSE
  ---------------------------------------------------------------
  Inherent Group                             7    8     4    3    7    5  1.15
  Meridian Capital Group LLC                 1    1     3    3    1    1  0.0
  PBUCC                                      6    8     4    6    3    8  5.95
  Pension Boards United Church of Christ     6    8     5    6    5    8  2.95
  The Rockefeller Foundation                 9    9    10    9    4    8  2.6

  Anchor MSE Loss : 2.53
  Anchor RMSE     : 1.59


In [21]:
final_loss = round((l1 * 0.50) + (l2 * 0.30) + (l3 * 0.20), 4)

print("=" * 50)
print("FINAL COMBINED LOSS REPORT")
print("=" * 50)
print(f"  Anchor Loss        (50%) : {l1}")
print(f"  Rule Violation     (30%) : {l2}")
print(f"  Stability Loss     (20%) : {l3}")
print(f"  ----------------------------")
print(f"  Final Combined Loss      : {final_loss}")
print()
if final_loss < 1.0:
    print("EXCELLENT -- Pipeline is well calibrated")
elif final_loss < 2.0:
    print("GOOD -- Minor issues, acceptable for production")
elif final_loss < 3.0:
    print("MODERATE -- Consider prompt tuning")
else:
    print("POOR -- Significant calibration issues")

FINAL COMBINED LOSS REPORT
  Anchor Loss        (50%) : 2.53
  Rule Violation     (30%) : 0.0028
  Stability Loss     (20%) : 0.22
  ----------------------------
  Final Combined Loss      : 1.3098

GOOD -- Minor issues, acceptable for production


## Cost Estimator


In [25]:
# GPT-4o pricing (as of 2025)
COST_PER_1K_INPUT_TOKENS  = 0.0025   # $2.50 per 1M input tokens
COST_PER_1K_OUTPUT_TOKENS = 0.0100   # $10.00 per 1M output tokens

total_input_tokens  = 0
total_output_tokens = 0

for r in results:
    total_input_tokens  += r.get('_input_tokens', 0)
    total_output_tokens += r.get('_output_tokens', 0)

total_tokens   = total_input_tokens + total_output_tokens
estimated_cost = (
    (total_input_tokens  / 1000 * COST_PER_1K_INPUT_TOKENS) +
    (total_output_tokens / 1000 * COST_PER_1K_OUTPUT_TOKENS)
)

print("=" * 45)
print("API COST ESTIMATE -- THIS RUN")
print("=" * 45)
print(f"  Orgs scored          : {len(results)}")
print(f"  Input tokens         : {total_input_tokens:,}")
print(f"  Output tokens        : {total_output_tokens:,}")
print(f"  Total tokens         : {total_tokens:,}")
print(f"  Estimated cost       : ${estimated_cost:.4f}")
print(f"  Cost per org         : ${estimated_cost/max(len(results),1):.4f}")
print()
print("  Scaling projection:")
cost_per_org = estimated_cost / max(len(results), 1)
for n in [100, 500, 1000, 5000]:
    print(f"    {n:>5} prospects -> ${cost_per_org * n:.2f}")
print("=" * 45)



API COST ESTIMATE -- THIS RUN
  Orgs scored          : 89
  Input tokens         : 99,955
  Output tokens        : 25,323
  Total tokens         : 125,278
  Estimated cost       : $0.5031
  Cost per org         : $0.0057

  Scaling projection:
      100 prospects -> $0.57
      500 prospects -> $2.83
     1000 prospects -> $5.65
     5000 prospects -> $28.27


## Sample Output


In [32]:
def build_output_table(results, scored_df):
    rows = []

    for _, row in scored_df.iterrows():
        org_name = row['Organization']
        org_data = next((r for r in results if r['org_name'] == org_name), {})

        # Combine reasoning into one Why sentence
        sector_r   = org_data.get('sector_fit_reasoning', '')
        halo_r     = org_data.get('halo_reasoning', '')
        emerging_r = org_data.get('emerging_fit_reasoning', '')
        why        = f"{sector_r} {halo_r} {emerging_r}".strip()

        rows.append({
            'Organization':  org_name,
            'Type':          row['Org Type'],
            'AUM':           org_data.get('aum_estimate', 'Unknown'),
            'Sector':        org_data.get('sector_fit_score', '?'),
            'Halo':          org_data.get('halo_score', '?'),
            'Emerging':      org_data.get('emerging_fit_score', '?'),
            'Composite':     row['Composite Score'],
            'Tier':          row['Tier'],
            'Confidence':    org_data.get('confidence', 'Unknown'),
            'Why':           why,
        })

    return pd.DataFrame(rows)


output_df = build_output_table(results, scored_df)

# Display full Why column without truncation
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 100)

output_df.head(20)

,Organization,Type,AUM,Sector,Halo,Emerging,Composite,Tier,Confidence,Why
0,The Schmidt Family Foundation,Foundation,Unknown,9,8,7,8.50,PRIORITY CLOSE,HIGH,"The foundation is deeply involved in sustainability and clean energy, aligning well with the ESG mandate. It also engages in impact investing, indicating some level of private credit allocation. T..."
1,Flat World Partners,Asset Manager,Unknown,9,7,6,8.15,PRIORITY CLOSE,MEDIUM,"Flat World Partners focuses on sustainable and impact investing with a strong emphasis on ESG factors, qualifying it well for a firm with a sustainability/ESG mandate. It also engages in activitie..."
2,Columbia Investment Management Company,Endowment,$13.8B,5,9,3,6.10,MODERATE FIT,HIGH,"Columbia Investment Management Company is an endowment manager, indicating allocation to alternatives including private credit; however, there is limited publicly available information explicitly ..."
3,Willett Advisors,Single Family Office,Unknown,6,7,7,5.75,MODERATE FIT,MEDIUM,"Willett Advisors is involved in alternatives exposure, aligning with its role as an LP, but there is limited specific information about a dedicated sustainability or private credit mandate. As the..."
4,AlTi Global,Multi-Family Office,Unknown,7,5,7,5.70,MODERATE FIT,MEDIUM,"AlTi Global is involved in wealth management with a focus on sustainability, indicating a potential alignment with ESG. However, specific private credit allocation is not strongly confirmed. AlTi ..."
5,BBR Partners,Multi-Family Office,$18B,6,5,7,5.65,MODERATE FIT,HIGH,"BBR Partners allocates to alternative investments, including private credit, but there is no clear evidence of a specific sustainability or ESG mandate. BBR Partners is a well-known multi-family o..."
6,Morgan Stanley Alternative Investment Partners,Fund of Funds,Unknown,5,9,4,5.65,MODERATE FIT,MEDIUM,"Morgan Stanley AIP is an LP that deals with alternative investments; however, there is limited explicit information regarding their commitment to sustainability or private credit allocation. Morga..."
7,Safra Group,Multi-Family Office,$200B,6,7,6,5.60,MODERATE FIT,HIGH,"While Safra Group is a prominent player with exposure to various investment sectors, there is no explicit evidence of a strong sustainability/ESG mandate in its investment strategy. The Safra Grou..."
8,HSBC,Fund of Funds,Estimated over $2.9 trillion,6,9,3,5.55,MODERATE FIT,HIGH,"HSBC has exposure to alternative investments through its fund of funds activities, but its strong sustainability/ESG focus is not heavily emphasized, resulting in a mid-range sector fit. HSBC is g..."
9,Bessemer Trust,Multi-Family Office,$107B,5,9,5,5.50,MODERATE FIT,HIGH,"Bessemer Trust acts as an LP and has exposure to alternative investments, however, there is limited evidence of a strong sustainability or ESG mandate. Bessemer Trust is widely recognized and resp..."
